In [2]:
import pandas as pd

# Load original file
df = pd.read_csv("learnit_courses.csv")

# Keep only the needed columns
course = pd.DataFrame()

# Assign ids from 0
course["id"] = range(len(df))

# period comes from Semester
course["period"] = df["Semester"]

# title comes from Course Name, but remove trailing " (Spring 2026)" etc.
course["title"] = (
    df["Course Name"]
    .str.replace(r"\s+\((Spring|Autumn)\s+\d{4}\)$", "", regex=True)
    .str.replace(",", " -")
)

course.to_csv("Course.csv", index=False)

In [3]:
import pandas as pd
import re

# Load files
course_df = pd.read_csv("Course.csv")                 # columns: id, period, title
learnit_df = pd.read_csv("learnit_courses.csv")       # original course export
person_df = pd.read_csv("person.csv")                 # columns: UUID, firstName, lastName, ...

# ---------- helpers ----------

def clean_title(title: str) -> str:
    title = str(title).strip()
    title = re.sub(r"\s+\((Spring|Autumn)\s+\d{4}\)$", "", title)
    return title.strip()

def clean_text(s: str) -> str:
    return str(s).strip().lower()

# ---------- normalize Course.csv ----------
course_df["title_clean"] = course_df["title"].apply(clean_title).apply(clean_text)
course_df["period_clean"] = course_df["period"].apply(clean_text)

# ---------- normalize learnit_courses.csv ----------
learnit_df["title_clean"] = learnit_df["Course Name"].apply(clean_title).apply(clean_text)
learnit_df["period_clean"] = learnit_df["Semester"].apply(clean_text)
learnit_df["teacher_clean"] = learnit_df["Teacher"].apply(clean_text)

# ---------- normalize person.csv ----------
person_df["full_name"] = (
    person_df["firstName"].astype(str).str.strip() + " " +
    person_df["lastName"].astype(str).str.strip()
)
person_df["full_name_clean"] = person_df["full_name"].apply(clean_text)

# ---------- match courses to Course.csv ----------
merged = learnit_df.merge(
    course_df[["id", "title_clean", "period_clean"]],
    on=["title_clean", "period_clean"],
    how="left"
)

# ---------- match teachers to person.csv ----------
merged = merged.merge(
    person_df[["UUID", "full_name_clean"]],
    left_on="teacher_clean",
    right_on="full_name_clean",
    how="left"
)

# ---------- build output ----------
result = merged[["UUID", "id"]].rename(columns={
    "UUID": "personUUID",
    "id": "courseId"
})

# keep only rows where both matches succeeded
result = result.dropna(subset=["personUUID", "courseId"])

# remove duplicates in case the source CSV has repeated rows
result = result.drop_duplicates()

# convert courseId to int
result["courseId"] = result["courseId"].astype(int)

# save
result.to_csv("PersonTeachesCourse.csv", index=False)

print(result.head(20))

                              personUUID  courseId
0   1b3cff7f-1549-4998-b52e-0ba9093e1d01         0
1   1b3cff7f-1549-4998-b52e-0ba9093e1d01         1
2   8785f18a-1bf3-4c63-8d8b-d3e5d516ca40         2
3   1b3cff7f-1549-4998-b52e-0ba9093e1d01         3
4   1b3cff7f-1549-4998-b52e-0ba9093e1d01         4
7   5c50bbc9-ed3d-417f-ba4d-e484eaf536b3         5
8   5f3b7745-99be-4d1b-8ce1-a413325c2008         6
9   1b3cff7f-1549-4998-b52e-0ba9093e1d01         7
10  1b3cff7f-1549-4998-b52e-0ba9093e1d01         8
11  1b3cff7f-1549-4998-b52e-0ba9093e1d01         9
12  1b3cff7f-1549-4998-b52e-0ba9093e1d01        10
13  1b3cff7f-1549-4998-b52e-0ba9093e1d01        11
36  a22e85da-82e3-410d-8ef1-31407362ba19        14
37  1b3cff7f-1549-4998-b52e-0ba9093e1d01        15
38  1b3cff7f-1549-4998-b52e-0ba9093e1d01        16
39  1b3cff7f-1549-4998-b52e-0ba9093e1d01        17
40  53c45a58-4c3c-424c-80a2-c39e733fd7c3        18
41  53c45a58-4c3c-424c-80a2-c39e733fd7c3        19
42  53c45a58-4c3c-424c-80a2-c39